# Mondi in miniatura: imparare sognando

Il codice del capitolo [«Mondi in miniatura: imparare sognando»](https://book.paithon.it/main/WorldModels/mondi-in-miniatura.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.%pip install -q torch torchvision

## Mondi in miniatura: imparare sognando

[Leggi la pagina](https://book.paithon.it/main/WorldModels/mondi-in-miniatura.html)


### I tre moduli in PyTorch


In [ ]:
import torchfrom torch import nnclass EncoderVAE(nn.Module):    """V: comprime un fotogramma 3x64x64 in un codice z di 32 numeri."""    def __init__(self, dim_z=32):        super().__init__()        self.conv = nn.Sequential(            nn.Conv2d(3, 32, 4, stride=2), nn.ReLU(),    # -> (32, 31, 31)            nn.Conv2d(32, 64, 4, stride=2), nn.ReLU(),   # -> (64, 14, 14)            nn.Conv2d(64, 128, 4, stride=2), nn.ReLU(),  # -> (128, 6, 6)            nn.Conv2d(128, 256, 4, stride=2), nn.ReLU(), # -> (256, 2, 2)            nn.Flatten(),                                # -> 1024        )        self.mu = nn.Linear(1024, dim_z)       # media del codice        self.logvar = nn.Linear(1024, dim_z)   # log-varianza del codice    def forward(self, x):                      # x: (B, 3, 64, 64)        h = self.conv(x)                       # (B, 1024)        mu, logvar = self.mu(h), self.logvar(h)        eps = torch.randn_like(mu)             # riparametrizzazione        return mu + torch.exp(0.5 * logvar) * eps   # z: (B, 32)class ModelloRNN(nn.Module):    """M: dato il codice e l'azione, predice il codice del passo dopo.    (Versione deterministica; il paper usa una miscela di gaussiane.)"""    def __init__(self, dim_z=32, dim_a=3, dim_h=256):        super().__init__()        self.lstm = nn.LSTM(dim_z + dim_a, dim_h, batch_first=True)        self.testa = nn.Linear(dim_h, dim_z)   # media del prossimo z    def forward(self, z, a, stato=None):       # z: (B, T, 32), a: (B, T, 3)        ingresso = torch.cat([z, a], dim=-1)   # (B, T, 35)        h, stato = self.lstm(ingresso, stato)  # h: (B, T, 256)        return self.testa(h), stato            # z predetto: (B, T, 32)class Controller(nn.Module):    """C: policy lineare da codice e memoria all'azione."""    def __init__(self, dim_z=32, dim_h=256, dim_a=3):        super().__init__()        self.lineare = nn.Linear(dim_z + dim_h, dim_a)   # 288*3+3 = 867    def forward(self, z, h):                   # z: (B, 32), h: (B, 256)        # azioni in [-1, 1]; gas e freno andrebbero poi riportati in [0, 1]        return torch.tanh(self.lineare(torch.cat([z, h], dim=-1)))

In [ ]:
V, M, C = EncoderVAE(), ModelloRNN(), Controller()print(sum(p.numel() for p in C.parameters()))   # 867: il pilota è minuscolox = torch.rand(1, 3, 64, 64)     # un fotogramma finto: batch 1, RGB, 64x64z = V(x).unsqueeze(1)            # (1, 1, 32): il codice, come sequenza di 1 passostato = None                     # memoria (h, c) della LSTM, vuota all'iniziofor t in range(10):              # dieci passi di sogno: nessun ambiente    a = torch.rand(1, 1, 3) * 2 - 1        # azione casuale in [-1, 1]    z, stato = M(z, a, stato)              # il codice sognato: (1, 1, 32)# il controller legge codice e memoria e restituisce i tre comandih = stato[0].squeeze(0)          # stato nascosto della LSTM: (1, 256)comandi = C(z.squeeze(1), h)     # (1, 3): sterzo, acceleratore, freno

## La via di LeCun: predire nello spazio delle idee

[Leggi la pagina](https://book.paithon.it/main/WorldModels/jepa.html)


### Una mini-JEPA in PyTorch


In [ ]:
import copyimport torchfrom torch import nntorch.manual_seed(0)DIM_PATCH, DIM_EMB = 16, 32N_PATCH, N_CONTESTO = 8, 6          # per scena: 6 patch visibili, 2 mascherate# Mappa fissa dal "contenuto" della scena all'aspetto delle patchPROIEZIONE = torch.randn(4, DIM_PATCH)def genera_batch(n=256):    """Ogni scena nasce da un contenuto latente comune alle sue 8 patch."""    contenuto = torch.randn(n, 1, 4)               # il "succo" della scena    patch = contenuto @ PROIEZIONE                 # come il succo appare    return patch + 0.25 * torch.randn(n, N_PATCH, DIM_PATCH)  # dettagli casuali# Encoder (l'allievo), predictor, ed encoder target (la copia lenta)encoder = nn.Sequential(    nn.Linear(DIM_PATCH, 64), nn.ReLU(), nn.Linear(64, DIM_EMB))predictor = nn.Sequential(    nn.Linear(DIM_EMB, 64), nn.ReLU(), nn.Linear(64, DIM_EMB))encoder_target = copy.deepcopy(encoder)for p in encoder_target.parameters():    p.requires_grad_(False)          # stop-gradient: il bersaglio non si allena@torch.no_grad()def aggiorna_target(tau=0.996):    """EMA: il target insegue lentamente l'encoder. È l'anti-collasso:    non ricevendo gradiente, non può 'mettersi d'accordo' con l'encoder    per appiattire tutti gli embedding sulla stessa costante."""    for p, p_t in zip(encoder.parameters(), encoder_target.parameters()):        p_t.mul_(tau).add_((1.0 - tau) * p)opt = torch.optim.Adam(    list(encoder.parameters()) + list(predictor.parameters()), lr=1e-3)for passo in range(1, 601):    patch = genera_batch()                          # (256, 8, 16)    # contesto -> embedding riassuntivo (media delle 6 patch visibili)    s_x = encoder(patch[:, :N_CONTESTO]).mean(dim=1)        # (256, 32)    # target -> embedding calcolato dalla copia lenta, senza gradiente    with torch.no_grad():        s_y = encoder_target(patch[:, N_CONTESTO:]).mean(dim=1)  # (256, 32)    s_y_pred = predictor(s_x)                       # predizione tra embedding    loss = nn.functional.mse_loss(s_y_pred, s_y)    # loss nello spazio latente    opt.zero_grad()    loss.backward()    opt.step()    aggiorna_target()                               # un passetto di EMA    if passo in (1, 100, 200, 400, 600):        # se gli embedding collassassero, questa varietà scenderebbe verso 0        varieta = s_y.std(dim=0).mean().item()        print(f"passo {passo}: loss {loss.item():.4f}  "              f"varietà degli embedding {varieta:.3f}")